In [4]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 64.2 MB/s eta 0:00:00


In [5]:
from gensim.models import Word2Vec
import re

In [6]:
# ── Prepare corpus ────────────────────────────────────────────────
corpus = [
    'the king rules the kingdom with power',
    'the queen rules the kingdom with elegance',
    'man is strong and powerful leader',
    'woman is intelligent and wise leader',
    'paris is the capital of france and a beautiful city',
    'london is the capital of england and a great city',
    'berlin is the capital of germany',
    'cat and dog are common household pets',
    'cats and dogs are popular pets in homes',
]




In [16]:
sentences = [
    re.sub(r'[^a-z ]', '', sent.lower()).split()
    for sent in corpus
]

print(sentences)

[['the', 'king', 'rules', 'the', 'kingdom', 'with', 'power'], ['the', 'queen', 'rules', 'the', 'kingdom', 'with', 'elegance'], ['man', 'is', 'strong', 'and', 'powerful', 'leader'], ['woman', 'is', 'intelligent', 'and', 'wise', 'leader'], ['paris', 'is', 'the', 'capital', 'of', 'france', 'and', 'a', 'beautiful', 'city'], ['london', 'is', 'the', 'capital', 'of', 'england', 'and', 'a', 'great', 'city'], ['berlin', 'is', 'the', 'capital', 'of', 'germany'], ['cat', 'and', 'dog', 'are', 'common', 'household', 'pets'], ['cats', 'and', 'dogs', 'are', 'popular', 'pets', 'in', 'homes']]


In [18]:
# ── Train Word2Vec ────────────────────────────────────────────────
model = Word2Vec(
    sentences=sentences,
    vector_size=100,    # Embedding dimensions (50-300 typical)
    window=5,           # Context window size
    min_count=1,        # Ignore words appearing < 1 time
    workers=4,          # CPU cores for training
    sg=1,               # 1=Skip-gram, 0=CBOW
    negative=5,         # Number of negative samples
    epochs=100,         # Training passes over corpus
    seed=42
)


In [9]:
# ── Explore the embeddings ───────────────────────────────────────
print('Vector for king:', model.wv['king'][:5], '...')  # First 5 dims


Vector for king: [ 0.00107791  0.01005743  0.00135898 -0.01010537  0.00709301] ...


In [10]:
# Find most similar words
print(model.wv.most_similar('king', topn=5))
# [('queen', 0.87), ('leader', 0.83), ('powerful', 0.76), ...]


[('and', 0.2900693714618683), ('cat', 0.26657330989837646), ('elegance', 0.2221856713294983), ('common', 0.20597927272319794), ('in', 0.16553141176700592)]


In [11]:
# The famous King - Man + Woman = Queen analogy!
result = model.wv.most_similar(
    positive=['king', 'woman'],
    negative=['man'],
    topn=3
)

In [12]:
print('king - man + woman =', result)  # Should be close to 'queen'


king - man + woman = [('power', 0.17773248255252838), ('cat', 0.16615231335163116), ('household', 0.16597013175487518)]


In [13]:
# Similarity scores
print('cat <-> dog:', model.wv.similarity('cat', 'dog'))    # high ~0.8
print('cat <-> king:', model.wv.similarity('cat', 'king'))  # low ~0.1

cat <-> dog: 0.30694506
cat <-> king: 0.26657334


In [14]:
# Doesn't fit:
print(model.wv.doesnt_match(['paris', 'london', 'berlin', 'cat']))

berlin


In [15]:
model.save('word2vec_model.bin')
loaded_model = Word2Vec.load('word2vec_model.bin')